# Docker Fundamentals: A Practical Learning Guide

## Overview
This notebook covers Docker concepts, architecture, and hands-on practice for developers with 2+ years of experience.

**What you'll learn:**
- Docker basics and containerization
- Working with images and containers
- Dockerfile creation and best practices
- Docker Compose for multi-container applications
- Networking and volumes
- Real-world examples

---
## Part 1: Docker Basics

### What is Docker?

Docker is a containerization platform that packages applications and their dependencies into isolated, reproducible units called **containers**.

### Key Concepts

| Concept | Definition |
|---------|------------|
| **Container** | A lightweight, standalone package with code, runtime, libraries, and settings |
| **Image** | A blueprint/template used to create containers (immutable) |
| **Dockerfile** | A text file with instructions to build an image |
| **Registry** | A repository to store and share images (e.g., Docker Hub) |
| **Volume** | Persistent data storage for containers |

### Docker vs Virtual Machines

```
Virtual Machine          Docker Container
├─ Full OS               ├─ Lightweight
├─ Heavy (GB)            ├─ Small (MB)
├─ Slow startup          ├─ Fast startup
├─ More isolation        ├─ Process-level isolation
└─ Resource intensive    └─ Efficient resource use
```

### Check Docker Installation

First, verify Docker is installed and running on your system.

In [1]:
import subprocess
import json

# Helper function to run Docker commands
def run_docker_command(cmd):
    """Execute Docker command and return output"""
    try:
        result = subprocess.run(
            f"docker {cmd}",
            shell=True,
            capture_output=True,
            text=True,
            timeout=10
        )
        return result.stdout if result.returncode == 0 else f"Error: {result.stderr}"
    except Exception as e:
        return f"Exception: {str(e)}"

# Check Docker version
print("Docker Version:")
print(run_docker_command("--version"))

# Check Docker info
print("\nDocker System Info:")
info = run_docker_command("info")
# Print first few lines
print('\n'.join(info.split('\n')[:10]))

Docker Version:
Docker version 29.3.1, build c2be9cc


Docker System Info:
Client:
 Version:    29.3.1
 Context:    desktop-linux
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.39.0
    Path:     C:\Program Files\Docker\cli-plugins\docker-agent.exe
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.20.2


---
## Part 2: Working with Docker Images

### What is a Docker Image?

An image is a **read-only template** that contains:
- Base OS/runtime (e.g., Python 3.9)
- Application code
- Dependencies
- Environment variables
- Default commands

Images are built from a **Dockerfile** using layers (each instruction creates a layer).

### Basic Image Commands

Let's explore common Docker image operations:

In [2]:
# List all images on your system
print("📦 Listing Docker Images:")
print("="*70)
images = run_docker_command("images")
print(images if images else "No images found. Let's pull some!")

📦 Listing Docker Images:
IMAGE             ID             DISK USAGE   CONTENT SIZE   EXTRA
postgres:latest   52e6ffd11fdd        649MB          168MB   U    



In [ ]:
# Pull an image from Docker Hub
print("Pulling Ubuntu image from Docker Hub...")
print("(This may take a moment)\n")
result = run_docker_command("pull ubuntu:22.04")
print(result)

In [3]:
# Inspect an image
print("🔍 Image Details:")
print("="*70)
inspect = run_docker_command("inspect ubuntu:22.04")
# Parse and pretty print
try:
    image_data = json.loads(inspect)
    info = image_data[0]
    print(f"ID: {info['Id'][:19]}...")
    print(f"Created: {info['Created']}")
    print(f"Size: {info['Size'] / (1024**2):.2f} MB")
    print(f"\nEnvironment Variables:")
    for env in info['Config']['Env'][:5]:
        print(f"  - {env}")
except:
    print("(Image inspection output shown above)")

🔍 Image Details:
(Image inspection output shown above)


---
## Part 3: Working with Containers

### What is a Container?

A **container** is a running instance of an image. Think of it as:
- Image = Class (blueprint)
- Container = Object (instance)

### Container Lifecycle

```
Created → Running → Paused → Stopped → Removed
```

### Creating and Running Containers

**Basic syntax:**
```bash
docker run [OPTIONS] IMAGE [COMMAND]
```

Common options:
- `-d` : Run in detached mode (background)
- `-it` : Interactive terminal
- `--name` : Assign a name to the container
- `-p` : Map ports (host:container)
- `-v` : Mount volumes
- `-e` : Set environment variables

In [ ]:
# Run a simple container (one-time execution)
print("🚀 Running a simple Ubuntu container:")
print("="*70)
result = run_docker_command("run --rm ubuntu:22.04 echo 'Hello from Docker!'")
print(result)

In [ ]:
# Run a container in the background
print("Starting a long-running container...\n")
result = run_docker_command("run -d --name my-sleep-container ubuntu:22.04 sleep 300")
container_id = result.strip()
print(f"Container ID: {container_id}")
print(f"(Container will sleep for 5 minutes)")

In [ ]:
# List running containers
print("\n📋 Running Containers:")
print("="*70)
containers = run_docker_command("ps")
print(containers)

In [ ]:
# Get container details
print("🔍 Container Logs:")
print("="*70)
logs = run_docker_command("logs my-sleep-container")
print(logs if logs else "(No output yet)")

In [ ]:
# Execute a command in a running container
print("Running command inside container:")
print("="*70)
result = run_docker_command("exec my-sleep-container ls -la /root")
print(result)

In [ ]:
# Stop and remove the container
print("Cleaning up...\n")
print("Stopping container:")
result = run_docker_command("stop my-sleep-container")
print(result)

print("Removing container:")
result = run_docker_command("rm my-sleep-container")
print(result)

print("\nVerifying removal:")
ps_output = run_docker_command("ps -a | grep my-sleep-container")
print("Container removed!" if not ps_output.strip() else ps_output)

---
## Part 4: Creating Custom Images with Dockerfile

### Understanding Dockerfiles

A Dockerfile is a text file with instructions to build an image. Each instruction creates a **layer**.

### Common Dockerfile Instructions

| Instruction | Purpose | Example |
|------------|---------|----------|
| `FROM` | Base image | `FROM python:3.9-slim` |
| `WORKDIR` | Set working directory | `WORKDIR /app` |
| `COPY` | Copy files from host | `COPY . /app` |
| `ADD` | Copy/extract files | `ADD archive.tar.gz /app` |
| `RUN` | Execute command | `RUN pip install -r requirements.txt` |
| `ENV` | Set environment variables | `ENV DEBUG=False` |
| `EXPOSE` | Document port | `EXPOSE 8000` |
| `CMD` | Default command | `CMD ["python", "app.py"]` |
| `ENTRYPOINT` | Override CMD | `ENTRYPOINT ["python"]` |

### Example 1: Simple Python Application

In [4]:
import os

# Create a temporary directory for our Docker project
project_dir = "/tmp/docker-python-app"
os.makedirs(project_dir, exist_ok=True)

# Create app.py
app_code = '''#!/usr/bin/env python3
import os
from datetime import datetime

print("=" * 50)
print("Docker Python Application")
print("=" * 50)
print(f"Current time: {datetime.now()}")
print(f"Environment: {os.environ.get('APP_ENV', 'production')}")
print(f"Debug mode: {os.environ.get('DEBUG', 'False')}")
print(f"Running inside container: {os.path.exists('/.dockerenv')}")
print("=" * 50)
'''

with open(f"{project_dir}/app.py", "w") as f:
    f.write(app_code)

print("✅ Created app.py")
print(app_code)

✅ Created app.py
#!/usr/bin/env python3
import os
from datetime import datetime

print("=" * 50)
print("Docker Python Application")
print("=" * 50)
print(f"Current time: {datetime.now()}")
print(f"Environment: {os.environ.get('APP_ENV', 'production')}")
print(f"Debug mode: {os.environ.get('DEBUG', 'False')}")
print(f"Running inside container: {os.path.exists('/.dockerenv')}")
print("=" * 50)



In [5]:
# Create Dockerfile
dockerfile_content = '''# Use official Python runtime as base image
FROM python:3.9-slim

# Set metadata
LABEL author="Docker Learning"
LABEL version="1.0"

# Set working directory inside container
WORKDIR /app

# Set environment variables
ENV APP_ENV=production
ENV DEBUG=False
ENV PYTHONUNBUFFERED=1

# Copy application code
COPY app.py .

# Run the application
CMD ["python", "app.py"]
'''

with open(f"{project_dir}/Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("✅ Created Dockerfile")
print(dockerfile_content)

✅ Created Dockerfile
# Use official Python runtime as base image
FROM python:3.9-slim

# Set metadata
LABEL author="Docker Learning"
LABEL version="1.0"

# Set working directory inside container
WORKDIR /app

# Set environment variables
ENV APP_ENV=production
ENV DEBUG=False
ENV PYTHONUNBUFFERED=1

# Copy application code
COPY app.py .

# Run the application
CMD ["python", "app.py"]



In [6]:
# Build the image
print("🔨 Building Docker image...\n")
result = run_docker_command(f"build -t my-python-app:1.0 {project_dir}")
print(result)

🔨 Building Docker image...

Exception: Command 'docker build -t my-python-app:1.0 /tmp/docker-python-app' timed out after 10 seconds


In [7]:
# Run the container from our custom image
print("🚀 Running the container...\n")
result = run_docker_command("run --rm -e DEBUG=True my-python-app:1.0")
print(result)

🚀 Running the container...

Docker Python Application
Current time: 2026-04-17 14:56:57.851822
Environment: production
Debug mode: True
Running inside container: True



In [8]:
# Check image layers
print("🔍 Image Layers (History):")
print("="*70)
history = run_docker_command("history my-python-app:1.0")
print(history)

🔍 Image Layers (History):
IMAGE          CREATED          CREATED BY                                      SIZE      COMMENT
8a59da4705a5   19 seconds ago   CMD ["python" "app.py"]                         0B        buildkit.dockerfile.v0
<missing>      19 seconds ago   COPY app.py . # buildkit                        12.3kB    buildkit.dockerfile.v0
<missing>      19 seconds ago   ENV PYTHONUNBUFFERED=1                          0B        buildkit.dockerfile.v0
<missing>      19 seconds ago   ENV DEBUG=False                                 0B        buildkit.dockerfile.v0
<missing>      19 seconds ago   ENV APP_ENV=production                          0B        buildkit.dockerfile.v0
<missing>      19 seconds ago   WORKDIR /app                                    8.19kB    buildkit.dockerfile.v0
<missing>      19 seconds ago   LABEL version=1.0                               0B        buildkit.dockerfile.v0
<missing>      19 seconds ago   LABEL author=Docker Learning                    0B   

### Example 2: Web Application with Dependencies

In [9]:
# Create Flask web app
web_app_dir = "/tmp/docker-flask-app"
os.makedirs(web_app_dir, exist_ok=True)

# Create Flask app
flask_code = '''from flask import Flask, jsonify
import os

app = Flask(__name__)

@app.route('/')
def hello():
    return jsonify({
        "message": "Hello from Docker!",
        "environment": os.environ.get("ENVIRONMENT", "development")
    })

@app.route('/health')
def health():
    return jsonify({"status": "healthy"}), 200

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
'''

with open(f"{web_app_dir}/app.py", "w") as f:
    f.write(flask_code)

print("✅ Created Flask app.py")

✅ Created Flask app.py


In [10]:
# Create requirements.txt
requirements = "Flask==2.3.0\nWerkzeug==2.3.0"

with open(f"{web_app_dir}/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ Created requirements.txt")
print(requirements)

✅ Created requirements.txt
Flask==2.3.0
Werkzeug==2.3.0


In [11]:
# Create Dockerfile for Flask app
flask_dockerfile = '''FROM python:3.9-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY app.py .

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
    CMD python -c "import requests; requests.get('http://localhost:5000/health')"

CMD ["python", "app.py"]
'''

with open(f"{web_app_dir}/Dockerfile", "w") as f:
    f.write(flask_dockerfile)

print("✅ Created Flask Dockerfile")

✅ Created Flask Dockerfile


In [12]:
# Build Flask image
print("🔨 Building Flask image...\n")
result = run_docker_command(f"build -t flask-app:1.0 {web_app_dir}")
print(result)

🔨 Building Flask image...



Exception in thread Thread-18 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\sande\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\sande\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\sande\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\sande\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 999: character maps to <undefined>


Exception: Command 'docker build -t flask-app:1.0 /tmp/docker-flask-app' timed out after 10 seconds


---
## Part 5: Dockerfile Best Practices

### 1. Use Specific Base Image Tags

❌ **Bad:**
```dockerfile
FROM python
```

✅ **Good:**
```dockerfile
FROM python:3.9-slim
```

### 2. Minimize Layers

❌ **Bad (multiple RUN commands):**
```dockerfile
RUN apt-get update
RUN apt-get install -y curl
RUN apt-get clean
```

✅ **Good (chained with &&):**
```dockerfile
RUN apt-get update && \
    apt-get install -y curl && \
    apt-get clean
```

### 3. Optimize Build Context

Create a `.dockerignore` file:
```
__pycache__
.git
.pytest_cache
*.pyc
.DS_Store
```

### 4. Use Multi-stage Builds

```dockerfile
# Build stage
FROM python:3.9 as builder
RUN pip install --user --no-cache-dir -r requirements.txt

# Runtime stage
FROM python:3.9-slim
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH
COPY app.py .
CMD ["python", "app.py"]
```

---
## Part 6: Port Mapping and Networking

### Port Mapping

Map container ports to host ports so you can access services.

**Syntax:**
```bash
docker run -p HOST_PORT:CONTAINER_PORT IMAGE
```

**Examples:**
- `-p 8000:5000` - Maps host port 8000 to container port 5000
- `-p 5000:5000` - Same port on both
- `-p 127.0.0.1:8000:5000` - Map only on localhost

In [ ]:
# Create a simple HTTP server image
http_server_dir = "/tmp/docker-http-server"
os.makedirs(http_server_dir, exist_ok=True)

# Create simple HTTP server
http_server_code = '''import http.server
import socketserver
import json
from datetime import datetime

class MyHandler(http.server.SimpleHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/api/status':
            self.send_response(200)
            self.send_header('Content-type', 'application/json')
            self.end_headers()
            data = {"status": "running", "timestamp": str(datetime.now())}
            self.wfile.write(json.dumps(data).encode())
        else:
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            html = "<h1>Hello from Docker!</h1><p>Server is running...</p>"
            self.wfile.write(html.encode())

PORT = 8000
with socketserver.TCPServer(("", PORT), MyHandler) as httpd:
    print(f"Server running on port {PORT}")
    httpd.serve_forever()
'''

with open(f"{http_server_dir}/server.py", "w") as f:
    f.write(http_server_code)

print("✅ Created HTTP server")

In [ ]:
# Create Dockerfile
http_dockerfile = '''FROM python:3.9-slim

WORKDIR /app
COPY server.py .

EXPOSE 8000
CMD ["python", "server.py"]
'''

with open(f"{http_server_dir}/Dockerfile", "w") as f:
    f.write(http_dockerfile)

print("✅ Created Dockerfile")

In [ ]:
# Build the image
print("🔨 Building HTTP server image...\n")
result = run_docker_command(f"build -t http-server:1.0 {http_server_dir}")
print(result)

In [ ]:
# Run container with port mapping
print("🚀 Running container with port mapping...\n")
result = run_docker_command("run -d --name http-server-container -p 8080:8000 http-server:1.0")
print(f"Container started: {result.strip()}")
print("\nAccess the server at: http://localhost:8080")

In [ ]:
# Verify the container is running
import time
time.sleep(2)  # Give server time to start

print("Checking container status...\n")
ps = run_docker_command("ps | grep http-server-container")
print(ps)

In [ ]:
# Cleanup
print("Cleaning up HTTP server container...\n")
run_docker_command("stop http-server-container")
run_docker_command("rm http-server-container")
print("✅ Container cleaned up")

---
## Part 7: Docker Volumes and Data Persistence

### Problem: Data Loss

By default, data inside a container is **lost** when the container is deleted.

### Solution: Volumes

Volumes allow **persistent data storage** outside the container.

### Volume Types

| Type | Location | Use Case |
|------|----------|----------|
| **Named** | Docker-managed | Databases, shared data |
| **Bind Mount** | Host filesystem | Development, config files |
| **tmpfs** | Memory (RAM) | Temporary data |

### Syntax

```bash
docker run -v VOLUME_NAME:/container/path IMAGE
docker run -v /host/path:/container/path IMAGE
```

In [ ]:
# Create a named volume
print("📦 Creating Docker volume...\n")
result = run_docker_command("volume create my-data-volume")
print(f"Volume created: {result.strip()}")

# List volumes
print("\n📋 Docker Volumes:")
print("="*50)
volumes = run_docker_command("volume ls")
print(volumes)

In [ ]:
# Run container with volume
print("Running container with volume...\n")
result = run_docker_command(
    "run -d --name volume-test -v my-data-volume:/data ubuntu:22.04 sleep 300"
)
container_id = result.strip()
print(f"Container: {container_id}")

# Create a file in the volume
print("\nCreating file in volume...")
result = run_docker_command(
    "exec volume-test bash -c 'echo \"Persistent Data\" > /data/test.txt'"
)
print("✅ File created in /data/test.txt")

# Verify file exists
result = run_docker_command("exec volume-test cat /data/test.txt")
print(f"\nFile content: {result.strip()}")

In [ ]:
# Stop and remove container
print("Stopping and removing container...\n")
run_docker_command("stop volume-test")
run_docker_command("rm volume-test")
print("✅ Container removed")

# Run another container with the same volume
print("\nRunning new container with same volume...")
result = run_docker_command(
    "run --rm -v my-data-volume:/data ubuntu:22.04 cat /data/test.txt"
)
print(f"\n📍 Data persists! Content: {result.strip()}")

In [ ]:
# Cleanup volume
print("\nCleaning up volume...")
result = run_docker_command("volume rm my-data-volume")
print(f"✅ {result.strip()}")

---
## Part 8: Docker Compose

### Why Docker Compose?

When you have multiple containers that need to work together, managing them individually is tedious. **Docker Compose** uses a YAML file to define and run multi-container applications.

### Compose File Structure

```yaml
version: '3.8'

services:
  web:
    image: nginx:latest
    ports:
      - "8080:80"
  
  db:
    image: postgres:13
    environment:
      POSTGRES_PASSWORD: secret

volumes:
  db_data:

networks:
  backend:
```

In [ ]:
# Create Docker Compose project directory
compose_dir = "/tmp/docker-compose-example"
os.makedirs(compose_dir, exist_ok=True)

# Create docker-compose.yml
compose_content = '''version: '3.8'

services:
  # Web service
  web:
    image: nginx:1.21-alpine
    container_name: compose-web
    ports:
      - "8080:80"
    volumes:
      - ./html:/usr/share/nginx/html
    networks:
      - app-network
    depends_on:
      - api
    healthcheck:
      test: ["CMD", "wget", "--quiet", "--tries=1", "--spider", "http://localhost/"]
      interval: 10s
      timeout: 5s
      retries: 3

  # API service
  api:
    image: python:3.9-slim
    container_name: compose-api
    command: python -m http.server 8000
    ports:
      - "8000:8000"
    working_dir: /app
    volumes:
      - ./api:/app
    environment:
      - ENV=production
    networks:
      - app-network

networks:
  app-network:
    driver: bridge
'''

with open(f"{compose_dir}/docker-compose.yml", "w") as f:
    f.write(compose_content)

print("✅ Created docker-compose.yml")
print(compose_content)

In [ ]:
# Create HTML directory and file
html_dir = f"{compose_dir}/html"
os.makedirs(html_dir, exist_ok=True)

html_content = '''<!DOCTYPE html>
<html>
<head>
    <title>Docker Compose Demo</title>
    <style>
        body { font-family: Arial; text-align: center; padding: 50px; }
        h1 { color: #0099ff; }
    </style>
</head>
<body>
    <h1>🐳 Docker Compose Demo</h1>
    <p>This is served from the web container!</p>
    <p>API server running on port 8000</p>
</body>
</html>
'''

with open(f"{html_dir}/index.html", "w") as f:
    f.write(html_content)

print("✅ Created HTML file")

In [ ]:
# Create API directory
api_dir = f"{compose_dir}/api"
os.makedirs(api_dir, exist_ok=True)

api_file_content = '''# Simple API file for serving
This is the API service running in Docker Compose.
'''

with open(f"{api_dir}/README.txt", "w") as f:
    f.write(api_file_content)

print("✅ Created API directory")

In [ ]:
# Validate compose file
print("🔍 Validating docker-compose.yml...\n")
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml config")
print("✅ Compose file is valid!")
# Show first 20 lines
print("\nValidated configuration (first 20 lines):")
print("\n".join(result.split("\n")[:20]))

In [ ]:
# Start services with compose
print("🚀 Starting services with Docker Compose...\n")
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml up -d")
print(result)

In [ ]:
import time
time.sleep(3)  # Wait for services to start

# Check running services
print("📋 Running Services (via Compose):")
print("="*70)
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml ps")
print(result)

In [ ]:
# View logs from specific service
print("📜 Logs from web service:")
print("="*70)
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml logs web")
print(result if result else "(No logs yet)")

In [ ]:
# Execute command in a service
print("Running command in api service:")
print("="*70)
result = run_docker_command(
    f"compose -f {compose_dir}/docker-compose.yml exec api ls -la /app"
)
print(result)

In [ ]:
# Stop all services
print("🛑 Stopping services...\n")
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml stop")
print(result)

# Remove all containers
print("Removing containers...")
result = run_docker_command(f"compose -f {compose_dir}/docker-compose.yml rm -f")
print("✅ Services cleaned up")

---
## Part 9: Networking Between Containers

### Container Communication

Containers can communicate with each other through:
1. **Same network** (DNS resolution by container name)
2. **Network aliases** (alternative names)
3. **Published ports** (via host)

### Network Types

| Type | Use Case |
|------|----------|
| **bridge** (default) | Container-to-container isolation |
| **host** | Direct host network access |
| **overlay** | Swarm/cluster communication |
| **none** | No networking |

In [ ]:
# List existing networks
print("📡 Docker Networks:")
print("="*70)
networks = run_docker_command("network ls")
print(networks)

In [ ]:
# Create a custom network
print("Creating custom network...\n")
result = run_docker_command("network create my-app-network")
print(f"Network created: {result.strip()[:20]}...")

In [ ]:
# Run containers on the same network
print("Starting two containers on same network...\n")

# Container 1
result1 = run_docker_command(
    "run -d --name web-container --network my-app-network "
    "-p 8080:80 nginx:latest"
)
print(f"Web container: {result1.strip()[:20]}...")

# Container 2
result2 = run_docker_command(
    "run -d --name app-container --network my-app-network "
    "ubuntu:22.04 sleep 300"
)
print(f"App container: {result2.strip()[:20]}...")

print("\n✅ Both containers on same network")

In [ ]:
import time
time.sleep(2)

# Test DNS resolution between containers
print("Testing DNS resolution between containers...\n")
result = run_docker_command(
    "exec app-container nslookup web-container"
)
print(result)

In [ ]:
# Ping between containers
print("Testing connectivity (ping)...\n")
result = run_docker_command(
    "exec app-container ping -c 2 web-container"
)
print(result)

In [ ]:
# Cleanup
print("Cleaning up...\n")
run_docker_command("stop web-container app-container")
run_docker_command("rm web-container app-container")
run_docker_command("network rm my-app-network")
print("✅ Cleaned up")

---
## Part 10: Docker Image Optimization

### Image Size Matters

Smaller images mean:
- Faster builds and pulls
- Lower storage costs
- Better security (smaller attack surface)

### Optimization Techniques

In [ ]:
# 1. Use slim/alpine base images
print("📊 Base Image Size Comparison:")
print("="*70)

images_to_compare = [
    "python:3.9",
    "python:3.9-slim",
    "python:3.9-alpine"
]

# Pull images
print("Pulling images for comparison...\n")
for img in images_to_compare:
    run_docker_command(f"pull {img}")

print("\nImage sizes:")
result = run_docker_command("images | grep python:3.9")
print(result)

In [ ]:
# 2. Create optimized Dockerfile
optimized_dir = "/tmp/docker-optimized"
os.makedirs(optimized_dir, exist_ok=True)

# Create requirements
requirements_content = """requests==2.28.0
click==8.1.0
"""

with open(f"{optimized_dir}/requirements.txt", "w") as f:
    f.write(requirements_content)

# Create app
app_content = '''import requests
import click

@click.command()
def main():
    print("Optimized Docker Image")
    print(f"requests version: {requests.__version__}")

if __name__ == "__main__":
    main()
'''

with open(f"{optimized_dir}/app.py", "w") as f:
    f.write(app_content)

print("✅ Created optimized app")

In [ ]:
# Create optimized Dockerfile (multi-stage)
optimized_dockerfile = '''# Stage 1: Builder
FROM python:3.9-slim as builder

WORKDIR /app
COPY requirements.txt .

# Install dependencies to a local directory
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: Runtime
FROM python:3.9-alpine

WORKDIR /app

# Copy only the installed packages
COPY --from=builder /root/.local /root/.local

# Set PATH
ENV PATH=/root/.local/bin:$PATH\
    PYTHONUNBUFFERED=1

# Copy application
COPY app.py .

CMD ["python", "app.py"]
'''

with open(f"{optimized_dir}/Dockerfile", "w") as f:
    f.write(optimized_dockerfile)

print("✅ Created optimized Dockerfile (multi-stage)")

In [ ]:
# Build optimized image
print("🔨 Building optimized image...\n")
result = run_docker_command(f"build -t optimized-app:1.0 {optimized_dir}")
print(result)

In [ ]:
# Compare image sizes
print("📊 Image Size Comparison:")
print("="*70)
result = run_docker_command("images | grep -E '(optimized-app|my-python-app|flask-app)'")
print(result)

---
## Part 11: Useful Docker Commands Reference

### Image Commands

```bash
docker images                    # List images
docker pull IMAGE               # Download image
docker push IMAGE               # Upload to registry
docker build -t TAG .           # Build image
docker rmi IMAGE                # Remove image
docker inspect IMAGE            # Get image details
docker history IMAGE            # Show image layers
```

### Container Commands

```bash
docker ps                       # List running containers
docker ps -a                    # List all containers
docker run IMAGE                # Create and start container
docker start CONTAINER          # Start stopped container
docker stop CONTAINER           # Stop container
docker rm CONTAINER             # Remove container
docker logs CONTAINER           # View logs
docker exec CONTAINER CMD       # Run command in container
docker attach CONTAINER         # Attach to running container
docker cp FILE CONTAINER:/path  # Copy file to container
docker stats CONTAINER          # Show resource usage
```

### Network Commands

```bash
docker network ls               # List networks
docker network create NAME      # Create network
docker network rm NAME          # Remove network
docker network inspect NAME     # Network details
```

### Volume Commands

```bash
docker volume ls                # List volumes
docker volume create NAME       # Create volume
docker volume rm NAME           # Remove volume
docker volume inspect NAME      # Volume details
```

### Cleanup Commands

```bash
docker system prune             # Remove unused objects
docker system df                # Disk usage
docker container prune          # Remove stopped containers
docker image prune              # Remove dangling images
docker volume prune             # Remove unused volumes
```

---
## Part 12: Practical Exercises

### Exercise 1: Create a Node.js Web Application

**Objective:** Build and run a Node.js app in Docker.

**Steps:**
1. Create a simple Express.js application
2. Create a Dockerfile
3. Build the image
4. Run the container with port mapping
5. Test the application

In [ ]:
# Exercise 1: Node.js Application
node_app_dir = "/tmp/docker-nodejs"
os.makedirs(node_app_dir, exist_ok=True)

# Create package.json
package_json = '''{
  "name": "docker-nodejs",
  "version": "1.0.0",
  "description": "Simple Node.js app in Docker",
  "main": "app.js",
  "scripts": {
    "start": "node app.js"
  },
  "dependencies": {
    "express": "4.18.0"
  }
}
'''

with open(f"{node_app_dir}/package.json", "w") as f:
    f.write(package_json)

print("✅ Created package.json")

In [ ]:
# Create Node.js app
node_app_code = '''const express = require('express');
const app = express();
const PORT = 3000;

app.get('/', (req, res) => {
    res.json({
        message: "Hello from Docker!",
        timestamp: new Date().toISOString(),
        container: "Node.js Express Server"
    });
});

app.get('/health', (req, res) => {
    res.json({ status: "healthy" });
});

app.listen(PORT, () => {
    console.log(`Server running on port ${PORT}`);
});
'''

with open(f"{node_app_dir}/app.js", "w") as f:
    f.write(node_app_code)

print("✅ Created app.js")

In [ ]:
# Create Dockerfile
node_dockerfile = '''FROM node:16-alpine

WORKDIR /app

COPY package.json .
RUN npm install --production

COPY app.js .

EXPOSE 3000

CMD ["npm", "start"]
'''

with open(f"{node_app_dir}/Dockerfile", "w") as f:
    f.write(node_dockerfile)

print("✅ Created Dockerfile")

In [ ]:
# Build the image
print("🔨 Building Node.js Docker image...\n")
result = run_docker_command(f"build -t nodejs-app:1.0 {node_app_dir}")
print(result if "Successfully built" in result else "Building... (first run may take time)")

### Exercise 2: Multi-Container Database Application

**Objective:** Create a docker-compose setup with a web app and database.

**Requirement:**
- Web service: Python Flask app
- Database service: SQLite with volume
- Services communicate via network
- Health checks enabled

In [ ]:
# Exercise 2: Flask + SQLite Compose Setup
db_app_dir = "/tmp/docker-db-app"
os.makedirs(db_app_dir, exist_ok=True)

# Create Flask app with database
db_app_code = '''from flask import Flask, jsonify
import sqlite3
import os
from datetime import datetime

app = Flask(__name__)
DB_PATH = '/data/app.db'

def init_db():
    if not os.path.exists('/data'):
        os.makedirs('/data')
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            created_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

@app.route('/api/users', methods=['GET'])
def get_users():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM users')
    users = cursor.fetchall()
    conn.close()
    return jsonify({"users": users})

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({"status": "healthy", "timestamp": datetime.now().isoformat()})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

with open(f"{db_app_dir}/app.py", "w") as f:
    f.write(db_app_code)

print("✅ Created Flask app with database")

In [ ]:
# Create requirements.txt
db_requirements = "Flask==2.3.0\nWerkzeug==2.3.0\n"

with open(f"{db_app_dir}/requirements.txt", "w") as f:
    f.write(db_requirements)

# Create Dockerfile
db_dockerfile = '''FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 5000

CMD ["python", "app.py"]
'''

with open(f"{db_app_dir}/Dockerfile", "w") as f:
    f.write(db_dockerfile)

print("✅ Created Dockerfile")

In [ ]:
# Create docker-compose.yml
db_compose = '''version: '3.8'

services:
  web:
    build: .
    container_name: db-app-web
    ports:
      - "5000:5000"
    volumes:
      - app_data:/data
    environment:
      - FLASK_ENV=production
    depends_on:
      - db
    healthcheck:
      test: ["CMD", "python", "-c", "import requests; requests.get('http://localhost:5000/api/health')"]
      interval: 10s
      timeout: 5s
      retries: 3

  db:
    image: sqlite:latest
    container_name: db-app-sqlite
    volumes:
      - app_data:/data

volumes:
  app_data:
'''

with open(f"{db_app_dir}/docker-compose.yml", "w") as f:
    f.write(db_compose)

print("✅ Created docker-compose.yml")

---
## Part 13: Troubleshooting Common Issues

### Issue 1: "Docker daemon is not running"

**Solution:**
```bash
# macOS
open /Applications/Docker.app

# Linux
sudo systemctl start docker

# Windows
# Start Docker Desktop from Applications
```

### Issue 2: "Permission denied while trying to connect to Docker daemon"

**Solution:**
```bash
# Add current user to docker group
sudo usermod -aG docker $USER
newgrp docker
```

### Issue 3: "Out of disk space"

**Solution:**
```bash
# Clean up unused objects
docker system prune -a

# Check disk usage
docker system df
```

### Issue 4: "Port already in use"

**Solution:**
```bash
# Use different port
docker run -p 8081:80 nginx

# Or kill process using port
lsof -ti:8080 | xargs kill -9
```

### Issue 5: "Container exits immediately"

**Debug:**
```bash
# Check logs
docker logs container-name

# Run with interactive terminal
docker run -it image-name /bin/bash
```

---
## Part 14: Docker Best Practices Summary

### Security

✅ **Do:**
- Run containers as non-root user
- Use specific base image versions
- Scan images for vulnerabilities
- Keep images updated

❌ **Don't:**
- Store secrets in Dockerfiles
- Use `latest` tag in production
- Run with `--privileged` unnecessarily

### Performance

✅ **Do:**
- Use `.dockerignore`
- Minimize layers
- Use multi-stage builds
- Cache dependencies

❌ **Don't:**
- Create massive images
- Use old base images
- Install unnecessary packages

### Development

✅ **Do:**
- Use docker-compose for local development
- Volume mount source code
- Use descriptive names
- Document your setup

❌ **Don't:**
- Hardcode configuration
- Mix application code with container logic
- Ignore error handling

---
## Part 15: System Cleanup and Final Assessment

Let's clean up all the Docker objects we created during this learning session.

In [ ]:
# Remove all unused Docker objects
print("🧹 System Cleanup...\n")

# Stop all running containers
print("Stopping all containers...")
run_docker_command("stop $(docker ps -q) 2>/dev/null || true")

# Remove all containers
print("Removing all containers...")
run_docker_command("rm $(docker ps -aq) 2>/dev/null || true")

# Prune system
print("Pruning unused objects...")
prune_result = run_docker_command("system prune -f")
print(prune_result)

print("✅ Cleanup complete")

In [ ]:
# Final system status
print("\n📊 Final Docker System Status:")
print("="*70)

print("\n🐳 Images:")
print(run_docker_command("images | head -5"))

print("\n📦 Containers:")
print(run_docker_command("ps -a | head -3"))

print("\n💾 Disk Usage:")
print(run_docker_command("system df"))

---
## Learning Summary

### What You've Learned

✅ **Fundamentals**
- What containers are and why they're useful
- Images vs containers
- Docker architecture

✅ **Practical Skills**
- Building custom Docker images
- Running and managing containers
- Working with volumes and persistence
- Container networking

✅ **Advanced Topics**
- Docker Compose for multi-container apps
- Image optimization
- Best practices and troubleshooting

### Next Steps

1. **Docker Registries**: Learn to push images to Docker Hub or private registries
2. **Container Orchestration**: Explore Kubernetes for managing multiple containers at scale
3. **CI/CD Integration**: Use Docker in GitHub Actions, GitLab CI, or Jenkins
4. **Production Deployment**: Deploy containerized apps to cloud platforms (AWS, GCP, Azure)
5. **Monitoring**: Set up container monitoring with Prometheus, Grafana, or ELK stack

### Resources

- **Official Docs**: https://docs.docker.com
- **Docker Hub**: https://hub.docker.com
- **Best Practices**: https://docs.docker.com/develop/dev-best-practices/

---

**Congratulations!** 🎉 You now have a solid understanding of Docker and are ready to use containers in your projects!